<a href="https://colab.research.google.com/github/TAUforPython/LLM_AI_agents/blob/main/lesson-9_DSPy_components.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSPy Example: Medical Appointment Assistant Optimization

DSPy: Language-Optimized Programming
Sweet Spot: DSPy optimizes prompts, instructions, and few-shot examples automatically
- Algorithm finds best prompt
- Auto-generates few-shot examples
- Rewrites instructions
- Needs: 20 test cases + metric
- Result: +20-30% quality


DSPy prompt optimization is an automated framework that replaces manual "prompt hacking" with algorithms that treat prompts as trainable parameters, tuning them to maximize performance metrics (like accuracy). It uses LLMs to generate, test, and select the best prompt variations based on your specific dataset, transforming prompt engineering into a systematic, data-driven science.

https://dspy.ai/learn/

In [ ]:
# Install required packages
!pip install dspy-ai openai -q
!pip install langchain_mistralai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5

In [ ]:
import dspy
import openai
import json
from typing import List, Tuple
import random

# DSPy example

In [ ]:
# Configure Mistral API
MODEL_NAME = 'mistral-small-latest' # or "mistral-large-latest"

import os
from google.colab import userdata

os.environ["MISTRAL_API_KEY"] = userdata.get("Mistral_API")

from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model=MODEL_NAME,
    temperature=0,
    max_retries=2,
)

In [ ]:
# Create a custom DSPy LM adapter for Mistral
class MistralAdapter(dspy.LM):
    def __init__(self, model_name):
        super().__init__(model=model_name)
        self.model_name = model_name
        self.history = []
        self.client = ChatMistralAI(
            model=model_name,
            temperature=0,
            max_retries=2,
        )

    def __call__(self, prompt=None, **kwargs):
        # Handle DSPy's call format
        if prompt is None:
            # This is likely a messages format call
            messages = kwargs.get('messages', [])
            if messages:
                # Convert messages to a single prompt string
                prompt = "\n".join([msg['content'] if isinstance(msg, dict) else str(msg) for msg in messages])

        # Format the prompt for Mistral
        messages = [{"role": "user", "content": prompt}]
        response = self.client.invoke(messages)
        answer = response.content

        # Store in history for debugging
        self.history.append({
            "prompt": prompt,
            "response": answer,
            "kwargs": kwargs
        })

        return [answer]

# Configure DSPy with Mistral adapter
mistral_lm = MistralAdapter(MODEL_NAME)
dspy.settings.configure(lm=mistral_lm)

# Define the signature for our medical appointment task
class MedicalAppointmentSignature(dspy.Signature):
    """
    Answer questions about medical appointments, policies, and procedures.
    """
    question = dspy.InputField(desc="The patient's question")
    answer = dspy.OutputField(desc="A helpful and accurate response to the patient's question")

# Define the module that will be optimized
class MedicalAppointmentModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(MedicalAppointmentSignature)

    def forward(self, question):
        result = self.generate_answer(question=question)
        return dspy.Prediction(answer=result.answer)

# Create test cases (20 examples as required by DSPy)
test_cases = [
    ("What is the cancellation policy?", "Cancellations must be made at least 24 hours before the appointment. Less than 24 hours will incur a $25 fee."),
    ("How do I refill my prescription?", "Prescription refills require 48-hour advance notice. Submit requests through the patient portal or by calling the pharmacy."),
    ("Do I need insurance verification?", "All patients must provide valid insurance information at registration. We verify coverage before each appointment."),
    ("Can I have a telemedicine appointment?", "Yes, telemedicine appointments require a stable internet connection. Coverage varies by insurance provider."),
    ("When are follow-up appointments scheduled?", "Follow-ups are typically scheduled within 2-4 weeks of the initial consultation depending on condition severity."),
    ("How do I access my lab results?", "Lab results are available through the patient portal within 2-5 business days."),
    ("What are the vaccination requirements?", "Routine vaccinations follow CDC guidelines. Travel vaccinations need 2-4 weeks advance planning."),
    ("How do I get a specialist referral?", "Specialist referrals require primary care physician approval and insurance pre-authorization."),
    ("What are the payment policies?", "Co-payments are collected at the time of service. Outstanding balances are due within 30 days."),
    ("How do I access my medical records?", "Requests require written authorization and valid ID. Records provided within 30 days."),
    ("What are my privacy rights?", "Patient information is protected under HIPAA. Consent required for information sharing."),
    ("What's the chronic disease management program?", "Structured management plans with regular monitoring and patient education sessions."),
    ("I need to cancel my appointment immediately", "Emergency cancellations are exempt from fees with proper documentation. Contact us immediately."),
    ("Can I get my prescription early?", "Refills require 48-hour notice. Controlled substances need in-person visits every 90 days."),
    ("What if my insurance changes?", "If insurance lapses, you're responsible for full cost of services. Update us immediately."),
    ("Are weekend appointments available?", "Weekend availability varies. Check with our scheduling system or call for options."),
    ("How do I update my contact information?", "Update contact information through the patient portal or by calling our office."),
    ("What happens if I miss my appointment?", "Missed appointments without notice may result in a no-show fee according to policy."),
    ("Can I bring someone to my appointment?", "Yes, companions are welcome. Let us know in advance if special accommodations needed."),
    ("How far in advance should I schedule?", "Schedule as early as possible. Urgent appointments may be available same-day or next-day.")
]

# Create training and validation sets
random.shuffle(test_cases)
train_set = test_cases[:15]  # Use 15 for training
val_set = test_cases[15:]    # Use 5 for validation

# Define a metric function for DSPy to optimize
def medical_metric(gold_label, pred, trace=None):
    """
    Evaluate the quality of the medical appointment assistant's responses
    """
    # Simple string similarity approach (in practice, you'd use more sophisticated metrics)
    gold_lower = gold_label.lower()
    pred_lower = pred.answer.lower()

    # Check if key information is present in the prediction
    if all(word in pred_lower for word in gold_lower.split()[:3]):  # At least first 3 words match conceptually
        return True
    elif any(word in pred_lower for word in gold_lower.split()):  # Partial match
        return True
    else:
        return False

# Create DSPy examples from test cases
train_examples = []
for question, answer in train_set:
    example = dspy.Example(question=question, answer=answer).with_inputs('question')
    train_examples.append(example)

# Initialize and compile the module
teleprompter = dspy.teleprompt.BootstrapFewShot(
    metric=medical_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=5
)

try:
    compiled_module = teleprompter.compile(
        student=MedicalAppointmentModule(),
        trainset=train_examples
    )

    # Evaluate the compiled module
    print("Evaluating optimized module...")
    correct = 0
    total = len(val_set)

    for question, expected_answer in val_set:
        prediction = compiled_module(question=question)
        is_correct = medical_metric(expected_answer, prediction)
        if is_correct:
            correct += 1
        print(f"Q: {question}")
        print(f"A: {prediction.answer}")
        print(f"Expected: {expected_answer}")
        print(f"Correct: {is_correct}")
        print("-" * 50)

    accuracy = correct / total
    print(f"Validation Accuracy: {accuracy:.2%}")

    # Demonstrate the optimized prompt and examples
    print("\n" + "="*60)
    print("OPTIMIZED PROMPT AND FEW-SHOT EXAMPLES")
    print("="*60)

    # Test with new questions to show improvement
    print("\nTesting with new questions:")
    new_questions = [
        "What's the policy if I need to cancel last minute?",
        "How do I update my insurance information?",
        "Can I schedule a follow-up appointment online?",
        "What if I can't afford my co-pay today?",
        "How secure is the patient portal?"
    ]

    for q in new_questions:
        response = compiled_module(question=q)
        print(f"Q: {q}")
        print(f"A: {response.answer}")
        print("-" * 30)

    # Performance comparison demonstration
    print("\n" + "="*60)
    print("PERFORMANCE COMPARISON")
    print("="*60)

    # Baseline module (without optimization)
    baseline_module = MedicalAppointmentModule()

    # Test the same questions on both modules
    improvement_count = 0
    for question, expected in val_set:
        baseline_pred = baseline_module(question=question)
        optimized_pred = compiled_module(question=question)

        baseline_correct = medical_metric(expected, baseline_pred)
        optimized_correct = medical_metric(expected, optimized_pred)

        if optimized_correct and not baseline_correct:
            improvement_count += 1
            print(f"IMPROVED: '{question[:30]}...' - Baseline: {baseline_correct}, Optimized: {optimized_correct}")

    print(f"\nImprovements seen in {improvement_count}/{len(val_set)} validation cases")
    print(f"This represents approximately {(improvement_count/len(val_set)*100):.0f}% improvement in accuracy!")

    # Show DSPy's optimization results
    print(f"\nDSPy optimized:")
    print(f"- Prompt instructions for medical appointment queries")
    print(f"- Selected best few-shot examples automatically")
    print(f"- Rewrote instructions to improve clarity")
    print(f"- Achieved {accuracy:.0%} accuracy on validation set")

except Exception as e:
    print(f"Error during compilation: {e}")
    print("Trying to run with baseline module instead...")

    # Run with baseline module if optimization fails
    print("\nRunning with baseline module:")
    correct = 0
    total = len(val_set)

    for question, expected_answer in val_set:
        prediction = MedicalAppointmentModule()(question=question)
        is_correct = medical_metric(expected_answer, prediction)
        if is_correct:
            correct += 1
        print(f"Q: {question}")
        print(f"A: {prediction.answer}")
        print(f"Expected: {expected_answer}")
        print(f"Correct: {is_correct}")
        print("-" * 50)

    accuracy = correct / total
    print(f"Baseline Validation Accuracy: {accuracy:.2%}")

 60%|██████    | 9/15 [00:20<00:13,  2.30s/it]


Error during compilation: 'Example' object has no attribute 'lower'
Trying to run with baseline module instead...

Running with baseline module:
Q: How do I get a specialist referral?
A: To get a specialist referral, follow these general steps:

1. **Consult Your Primary Care Physician (PCP):**
   Start by discussing your symptoms or health concerns with your PCP. They will evaluate your condition and determine if a referral to a specialist is necessary. Your PCP may have specific specialists they prefer to work with.

2. **Check Your Insurance Requirements:**
   Review your health insurance policy to understand whether a referral is required before seeing a specialist. Some insurance plans mandate referrals from a PCP to cover the cost of specialist visits. If you're unsure, contact your insurance provider for details.

3. **Obtain the Referral:**
   If your insurance requires one, your PCP will provide a formal referral. This document typically includes the specialist's name, reason 

# DSPy Framework Demonstration: Medical Appointment Assistant


DSPy Framework Components:
- Signature (WHAT): Defines inputs and outputs
- Module (HOW): How to call LLM with specific logic
- Metric (HOW to score): Evaluation function for quality
- Optimizer: Finds best prompts, instructions, and examples


In [15]:

from typing import List

# Configure with a local model or API
# Using a mock model for this example since we're focusing on DSPy concepts
class MockLM(dspy.LM):
    def __init__(self):
        super().__init__("mock")
        self.history = []

    def __call__(self, *args, **kwargs):
        # Handle different call patterns
        prompt = None

        # Check if first argument is the prompt
        if args:
            if isinstance(args[0], str):
                prompt = args[0]
            elif isinstance(args[0], list) and len(args[0]) > 0:
                # Handle list of messages
                first_arg = args[0]
                if isinstance(first_arg, list):
                    # If it's a list of messages
                    if len(first_arg) > 0 and isinstance(first_arg[0], dict):
                        prompt = first_arg[0].get('content', str(first_arg[0]))
                    else:
                        prompt = str(first_arg)
                else:
                    prompt = str(first_arg)
        elif 'prompt' in kwargs:
            prompt = kwargs['prompt']
        elif 'messages' in kwargs:
            messages = kwargs['messages']
            if isinstance(messages, list) and len(messages) > 0:
                if isinstance(messages[0], dict):
                    prompt = messages[0].get('content', str(messages[0]))
                else:
                    prompt = str(messages[0])

        if prompt is None:
            prompt = str(args) if args else str(kwargs)

        # Mock response based on prompt content
        if "cancellation" in prompt.lower():
            response = "Our cancellation policy requires 24 hours notice. Cancellations with less notice may incur a fee."
        elif "prescript" in prompt.lower():
            response = "Prescription refills require 48-hour advance notice through our patient portal."
        elif "insurance" in prompt.lower():
            response = "We accept most major insurance plans. Please verify your coverage before your appointment."
        elif "telemedicine" in prompt.lower():
            response = "Yes, we offer telemedicine appointments for follow-up visits and certain consultations."
        elif "lab results" in prompt.lower():
            response = "Lab results are available through our secure patient portal within 2-3 business days."
        elif "schedule" in prompt.lower() or "appointment" in prompt.lower():
            response = "You can schedule appointments online through our patient portal or by calling our office."
        elif "emergency" in prompt.lower():
            response = "For emergencies, please call 911 or go to the nearest emergency room immediately."
        elif "late" in prompt.lower():
            response = "If you're running late, please call us immediately. We may need to reschedule."
        else:
            response = "Thank you for your question. Please contact our office for assistance with your medical appointment needs."

        self.history.append({
            "prompt": prompt,
            "response": response,
            "kwargs": kwargs
        })

        return [response]

# Configure DSPy with the mock LM
dspy.settings.configure(lm=MockLM())

# ============================================================================
# 1. SIGNATURE (WHAT) - Defines the task structure
# ============================================================================

class MedicalAppointmentSignature(dspy.Signature):
    """
    Defines WHAT the model should do:
    - Take a patient question as input
    - Produce a helpful medical appointment response as output
    """
    patient_question = dspy.InputField(
        desc="The specific question asked by the patient about appointments, policies, or procedures"
    )
    helpful_response = dspy.OutputField(
        desc="A clear, accurate, and helpful response addressing the patient's question"
    )

# Another signature example for multi-step reasoning
class MedicalAppointmentWithReasoning(dspy.Signature):
    """
    Signature with intermediate reasoning steps
    """
    patient_question = dspy.InputField(desc="The patient's question")
    thoughts = dspy.OutputField(desc="Chain of thought reasoning before final answer")
    final_answer = dspy.OutputField(desc="The final helpful response")

# ============================================================================
# 2. MODULE (HOW) - Implements the logic for calling LLM
# ============================================================================

class SimpleMedicalAssistant(dspy.Module):
    """
    Module that implements HOW to solve the task:
    - Uses a single LLM call to generate response
    """
    def __init__(self):
        super().__init__()
        self.generator = dspy.ChainOfThought(MedicalAppointmentSignature)

    def forward(self, question):
        result = self.generator(patient_question=question)
        return dspy.Prediction(response=result.helpful_response)

class ReasoningMedicalAssistant(dspy.Module):
    """
    Module with reasoning steps:
    - Uses Chain of Thought to break down the problem
    """
    def __init__(self):
        super().__init__()
        self.reasoner = dspy.ChainOfThought(MedicalAppointmentWithReasoning)

    def forward(self, question):
        result = self.reasoner(patient_question=question)
        return dspy.Prediction(
            thoughts=result.thoughts,
            response=result.final_answer
        )

class RAGMedicalAssistant(dspy.Module):
    """
    Module with Retrieval-Augmented Generation:
    - Retrieves relevant context first
    - Then generates response based on context
    """
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(SignatureWithRetrieval)

    def forward(self, context, question):
        result = self.generate(context=context, question=question)
        return dspy.Prediction(response=result.answer)

# Signature for RAG approach
class SignatureWithRetrieval(dspy.Signature):
    context = dspy.InputField(desc="Retrieved information relevant to the question")
    question = dspy.InputField(desc="The question to answer")
    answer = dspy.OutputField(desc="Answer based on the context")

# ============================================================================
# 3. METRIC (HOW to score) - Evaluation function
# ============================================================================

def medical_quality_metric(example, pred, trace=None):
    """
    Custom metric to evaluate medical appointment responses
    Returns True if response meets quality criteria, False otherwise
    """
    question = example.question
    expected_answer = example.answer
    predicted_answer = pred.response

    # Basic checks
    if not predicted_answer or len(predicted_answer.strip()) < 5:
        return False

    # Check if response addresses the question type
    question_lower = question.lower()
    answer_lower = predicted_answer.lower()

    # Medical-specific checks
    if "cancel" in question_lower and "cancel" not in answer_lower:
        return False
    if "prescript" in question_lower and "prescript" not in answer_lower:
        return False
    if "insurance" in question_lower and "insurance" not in answer_lower:
        return False

    # Check for helpfulness indicators
    helpful_indicators = ["please", "contact", "call", "visit", "information", "policy"]
    is_helpful = any(indicator in answer_lower for indicator in helpful_indicators)

    return is_helpful

def accuracy_metric(example, pred, trace=None):
    """
    Simple accuracy metric based on keyword matching
    """
    expected = example.answer.lower()
    predicted = pred.response.lower()

    # Check if key terms from expected answer appear in prediction
    expected_terms = expected.split()[:5]  # First 5 terms as proxy for main concepts
    matched_terms = sum(1 for term in expected_terms if term in predicted)

    # Consider accurate if at least half the key terms are present
    return matched_terms >= len(expected_terms) / 2

# ============================================================================
# 4. OPTIMIZER - Finds best prompts, instructions, and examples
# ============================================================================

# Sample data for optimization
sample_data = [
    dspy.Example(
        question="What is your cancellation policy?",
        answer="Our cancellation policy requires 24 hours notice. Cancellations with less notice may incur a fee."
    ).with_inputs('question'),
    dspy.Example(
        question="How do I refill my prescription?",
        answer="Prescription refills require 48-hour advance notice through our patient portal."
    ).with_inputs('question'),
    dspy.Example(
        question="Do you offer telemedicine appointments?",
        answer="Yes, we offer telemedicine appointments for follow-up visits and certain consultations."
    ).with_inputs('question'),
    dspy.Example(
        question="How can I access my lab results?",
        answer="Lab results are available through our secure patient portal within 2-3 business days."
    ).with_inputs('question'),
    dspy.Example(
        question="What insurance do you accept?",
        answer="We accept most major insurance plans. Please verify your coverage before your appointment."
    ).with_inputs('question'),
    dspy.Example(
        question="Can I schedule online?",
        answer="Yes, you can schedule appointments online through our patient portal or mobile app."
    ).with_inputs('question'),
    dspy.Example(
        question="What if I'm late for my appointment?",
        answer="If you're running late, please call us immediately. We may need to reschedule."
    ).with_inputs('question'),
    dspy.Example(
        question="How do I update my contact information?",
        answer="You can update your contact information through the patient portal or by calling our office."
    ).with_inputs('question'),
    dspy.Example(
        question="Are weekend appointments available?",
        answer="We offer limited weekend hours. Please check our website for availability."
    ).with_inputs('question'),
    dspy.Example(
        question="What should I bring to my appointment?",
        answer="Please bring your ID, insurance card, and a list of current medications."
    ).with_inputs('question'),
    dspy.Example(
        question="How early should I arrive?",
        answer="Please arrive 15 minutes early for new patients and 5 minutes early for follow-ups."
    ).with_inputs('question'),
    dspy.Example(
        question="Can I bring a family member?",
        answer="Yes, family members and caregivers are welcome to accompany you to appointments."
    ).with_inputs('question'),
    dspy.Example(
        question="What if I need to reschedule?",
        answer="You can reschedule online or by calling our office at least 24 hours in advance."
    ).with_inputs('question'),
    dspy.Example(
        question="Do you have parking?",
        answer="Yes, we have free parking available in our lot adjacent to the building."
    ).with_inputs('question'),
    dspy.Example(
        question="How long are typical appointments?",
        answer="Initial consultations are 30-45 minutes; follow-ups are typically 15-20 minutes."
    ).with_inputs('question'),
    dspy.Example(
        question="What if I have an emergency?",
        answer="For emergencies, please call 911 or go to the nearest emergency room immediately."
    ).with_inputs('question'),
    dspy.Example(
        question="Can I get my results over the phone?",
        answer="Basic results can be provided over the phone, but detailed discussions require an appointment."
    ).with_inputs('question'),
    dspy.Example(
        question="Do you offer same-day appointments?",
        answer="Same-day appointments are available for urgent matters, subject to provider availability."
    ).with_inputs('question'),
    dspy.Example(
        question="How do I prepare for my visit?",
        answer="Review your medications, bring relevant medical records, and arrive prepared with questions."
    ).with_inputs('question'),
    dspy.Example(
        question="What if I don't speak English?",
        answer="We provide interpreter services. Please indicate your language needs when scheduling."
    ).with_inputs('question')
]

# Split data for training and validation
random.seed(42)
random.shuffle(sample_data)
train_data = sample_data[:15]
val_data = sample_data[15:]

# Demonstrate different optimizers
print("DSPy FRAMEWORK DEMONSTRATION")
print("=" * 50)

print("\n1. SIGNATURES (WHAT)")
print("- Define inputs and outputs for LLM tasks")
print("- Examples: MedicalAppointmentSignature, MedicalAppointmentWithReasoning")
print("- Specify descriptions for better prompting")

print("\n2. MODULES (HOW)")
print("- Implement logic for calling LLMs")
print("- Examples: SimpleMedicalAssistant, ReasoningMedicalAssistant")
print("- Can include Chain of Thought, RAG, or other patterns")

print("\n3. METRICS (HOW to score)")
print("- Evaluate response quality")
print("- Examples: medical_quality_metric, accuracy_metric")
print("- Return True/False or score for optimization")

print("\n4. OPTIMIZERS - Finding best configurations")
print("- Demonstrating BootstrapFewShot optimizer...")

# BootstrapFewShot optimizer - finds best few-shot examples
bootstrap_optimizer = dspy.teleprompt.BootstrapFewShot(
    metric=medical_quality_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=5
)

# Compile the module with optimization
try:
    optimized_module = bootstrap_optimizer.compile(
        student=ReasoningMedicalAssistant(),
        trainset=train_data
    )

    print("\n✅ Optimization successful!")
    print("- Automatically selected best few-shot examples")
    print("- Optimized prompt instructions")
    print("- Improved performance based on metric")

    # Test the optimized module
    print(f"\nTesting optimized module on {len(val_data)} validation examples:")
    correct = 0
    for example in val_data:
        pred = optimized_module(example.question)
        is_correct = medical_quality_metric(example, pred)
        if is_correct:
            correct += 1
        print(f"Q: {example.question[:50]}...")
        print(f"A: {pred.response[:80]}... {'✅' if is_correct else '❌'}")

    print(f"\nAccuracy: {correct}/{len(val_data)} = {correct/len(val_data)*100:.1f}%")

except Exception as e:
    print(f"❌ Optimization failed: {e}")
    print("\nFalling back to manual demonstration...")

    # Show manual module usage
    manual_module = ReasoningMedicalAssistant()
    print("\nTesting manual module:")
    for i, example in enumerate(val_data[:3]):  # Test first 3 examples
        try:
            pred = manual_module(example.question)
            print(f"Q{i+1}: {example.question}")
            print(f"A: {pred.response}")
            print("-" * 40)
        except Exception as e:
            print(f"Error processing example {i+1}: {e}")

print("\n" + "=" * 50)
print("DSPy ADVANTAGES:")
print("• Automatic prompt optimization")
print("• Few-shot example selection")
print("• Instruction rewriting")
print("• Modular, composable components")
print("• Metric-driven improvement")
print("=" * 50)

DSPy FRAMEWORK DEMONSTRATION

1. SIGNATURES (WHAT)
- Define inputs and outputs for LLM tasks
- Examples: MedicalAppointmentSignature, MedicalAppointmentWithReasoning
- Specify descriptions for better prompting

2. MODULES (HOW)
- Implement logic for calling LLMs
- Examples: SimpleMedicalAssistant, ReasoningMedicalAssistant
- Can include Chain of Thought, RAG, or other patterns

3. METRICS (HOW to score)
- Evaluate response quality
- Examples: medical_quality_metric, accuracy_metric
- Return True/False or score for optimization

4. OPTIMIZERS - Finding best configurations
- Demonstrating BootstrapFewShot optimizer...


  0%|          | 0/15 [00:00<?, ?it/s]2026/04/17 17:08:27 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'question': "What if I don't speak English?", 'answer': 'We provide interpreter services. Please indicate your language needs when scheduling.'}) (input_keys={'question'}) with <function medical_quality_metric at 0x7a89b8d22f20> due to LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: Thank you for your question. Please contact our office for assistance with your medical appointment needs. 

Expected to find output fields in the LM response: [reasoning, thoughts, final_answer] 

.
  7%|▋         | 1/15 [00:00<00:01,  9.31it/s]2026/04/17 17:08:27 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'question': 'Can I schedule online?', 'answer': 'Yes, you can schedule appointments online through our patient portal or mobile app.'}) (input_keys={'question'}) 

❌ Optimization failed: LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: Thank you for your question. Please contact our office for assistance with your medical appointment needs. 

Expected to find output fields in the LM response: [reasoning, thoughts, final_answer] 



Falling back to manual demonstration...

Testing manual module:
Error processing example 1: LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: Thank you for your question. Please contact our office for assistance with your medical appointment needs. 

Expected to find output fields in the LM response: [reasoning, thoughts, final_answer] 


Error processing example 2: LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: Thank you for your question. Please contact our office for assistance with your medical appointmen